In [0]:
# ============================================
# RAILWAY DATA ENGINEERING PROJECT
# SILVER LAYER
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read Bronze Delta table
df_silver = spark.table(
    "railway_data_engineering.bronze.railway_bronze"
)

display(df_silver.limit(10))

In [0]:
# Check total records
print("Total Bronze Records:", df_silver.count())

In [0]:
# Check missing values in each column

from pyspark.sql.functions import sum as spark_sum

null_counts = df_silver.select([
    spark_sum(
        when(col(c).isNull() | (trim(col(c)) == ""), 1).otherwise(0)
    ).alias(c)
    for c in df_silver.columns
])

display(null_counts)

In [0]:
# Remove duplicate railway records

before_count = df_silver.count()

df_silver = df_silver.dropDuplicates([
    "Train_No",
    "Train_Name",
    "Source_Station_Name",
    "Destination_Station_Name",
    "days"
])

after_count = df_silver.count()

print("Records before duplicate removal:", before_count)
print("Records after duplicate removal :", after_count)
print("Duplicates removed              :", before_count - after_count)

In [0]:
# Standardize station names

df_silver = df_silver \
    .withColumn(
        "Source_Station_Name",
        upper(trim(regexp_replace(col("Source_Station_Name"), r"\s+", " ")))
    ) \
    .withColumn(
        "Destination_Station_Name",
        upper(trim(regexp_replace(col("Destination_Station_Name"), r"\s+", " ")))
    )

display(
    df_silver.select(
        "Source_Station_Name",
        "Destination_Station_Name"
    ).limit(10)
)

In [0]:
# Clean and standardize operating days
# Remove unwanted trailing 'd' from day values

df_silver = df_silver.withColumn(
    "days_clean",
    initcap(
        trim(
            regexp_replace(col("days"), "d$", "")
        )
    )
)
display(
    df_silver.select("days", "days_clean")
             .distinct()
             .orderBy("days_clean")
)

In [0]:
# Create numerical day representation

df_silver = df_silver.withColumn(
    "day_number",
    when(col("days_clean") == "Monday", 1)
    .when(col("days_clean") == "Tuesday", 2)
    .when(col("days_clean") == "Wednesday", 3)
    .when(col("days_clean") == "Thursday", 4)
    .when(col("days_clean") == "Friday", 5)
    .when(col("days_clean") == "Saturday", 6)
    .when(col("days_clean") == "Sunday", 7)
)

display(
    df_silver
    .select("days_clean", "day_number")
    .distinct()
    .orderBy("day_number")
)

In [0]:
# Create Weekday / Weekend classification

df_silver = df_silver.withColumn(
    "day_type",
    when(
        col("days_clean").isin("Saturday", "Sunday"),
        "Weekend"
    ).otherwise("Weekday")
)

display(
    df_silver
    .select("days_clean", "day_type")
    .distinct()
    .orderBy("days_clean")
)

In [0]:
# Create Train Category

df_silver = df_silver.withColumn(
    "Train_Category",
    col("day_type")
)

display(
    df_silver
    .select("days_clean", "day_type", "Train_Category")
    .distinct()
    .orderBy("day_number")
)

In [0]:
# ============================================
# SILVER DATA QUALITY VALIDATION
# ============================================

print("Silver record count:", df_silver.count())

In [0]:

null_check = df_silver.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_silver.columns
])

display(null_check)

In [0]:
# Check whether any invalid day values remain

valid_days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

invalid_days = df_silver.filter(
    ~col("days_clean").isin(valid_days)
)

print("Invalid day records:", invalid_days.count())

display(invalid_days)

In [0]:
# Validate Weekday vs Weekend classification

display(
    df_silver
    .groupBy("day_type")
    .count()
    .orderBy("day_type")
)

In [0]:
# Validate day-wise distribution

display(
    df_silver
    .groupBy("days_clean", "day_number")
    .count()
    .orderBy("day_number")
)

In [0]:
# Check for duplicate business records

duplicate_check = df_silver.groupBy(
    "Train_No",
    "Train_Name",
    "Source_Station_Name",
    "Destination_Station_Name",
    "days_clean"
).count().filter(col("count") > 1)

print("Duplicate business records:", duplicate_check.count())

display(duplicate_check)

In [0]:
# ============================================
# WRITE SILVER DELTA TABLE
# ============================================

silver_table = "railway_data_engineering.silver.railway_cleaned"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("✅ Silver Delta table created successfully!")
print("Table:", silver_table)

In [0]:
# Verify Silver Delta table

silver_check = spark.table(
    "railway_data_engineering.silver.railway_cleaned"
)

print("Silver record count:", silver_check.count())

display(silver_check.limit(10))